In [32]:
import joblib
import os
import time
import csv
from typing import Literal
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_log_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.compose import TransformedTargetRegressor
from sklearn.compose import make_column_selector
from sklearn.preprocessing import OrdinalEncoder

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from pre_processamento.engenharia_features import EngenhariaFeatures
from pre_processamento.transformacao_logaritmica import TransformacaoLogaritmica
from pre_processamento.valores_faltantes import ValoresFaltantes

# df_treino = pd.read_csv("./data/treino.csv",keep_default_na=False)
df_treino = pd.read_csv("./data/treino.csv")
# df_treino = pd.read_csv("./data/treino_norm.csv")

arquivo = "comparacao_modelos_treino.csv"

if os.path.exists(arquivo):
    os.remove(arquivo)

def validar_ajuste(modelo, X_treino, Y_treino, X_teste, Y_teste):

    # previsões
    pred_treino = modelo.predict(X_treino)
    pred_teste = modelo.predict(X_teste)

    # métricas
    r2_treino = r2_score(Y_treino, pred_treino)
    r2_teste = r2_score(Y_teste, pred_teste)

    rmse_treino = root_mean_squared_error(Y_treino, pred_treino)
    rmse_teste = root_mean_squared_error(Y_teste, pred_teste)

    print("\n===== VALIDAÇÃO DO MODELO =====")
    print(f"R² treino: {r2_treino:.4f}")
    print(f"R² teste:  {r2_teste:.4f}")

    print(f"RMSE treino: {rmse_treino:.4f}")
    print(f"RMSE teste:  {rmse_teste:.4f}")

    diferenca = r2_treino - r2_teste

    # diagnóstico
    if r2_treino < 0.7 and r2_teste < 0.7:
        print("Possível UNDERFITTING")
    elif diferenca > 0.1:
        print("Possível OVERFITTING")
    else:
        print("Modelo aparentemente bem ajustado")

def transformar_dolar(valor:int):
    return np.expm1(valor)

def salvar_modelo(modelo,caminho:str):
    
    os.makedirs("./modelos",exist_ok=True)
    
    joblib.dump(modelo,caminho)

def criar_csv_comparacao(modelo, rmsle, rmse, mae, r2, tempo_s):
    arquivo_existe = os.path.exists(arquivo)

    with open(arquivo, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not arquivo_existe:
            writer.writerow(["modelo", "rmsle", "rmse", "mae", "r2","tempo_s"])
            writer.writerow(["linear regression (BASELINE)", "0.17543", "36,061.40", "22,186.99", "0.83046","NA"])

        writer.writerow([modelo, f"{rmsle:.5f}", f"{rmse:.2f}", f"{mae:.2f}", f"{r2:.2f}", f"{tempo_s:.4f}"])

In [33]:
def comparar_baseline(alvo: Literal["RMSLE", "RMSE", "MAE", "R2"], metrica):
    RMSLE_META = 0.17543
    RMSE_META = 36061.40
    MAE_META = 22186.99
    R2_META = 0.83046

    match alvo:
        case "RMSLE":
            return metrica < RMSLE_META
        case "RMSE":
            return metrica < RMSE_META 
        case "MAE":
            return metrica < MAE_META  
        case "R2":
            return metrica > R2_META

# RandomForestRegressor

In [34]:
def treinar_random_forest(df:pd.DataFrame):
    X = df.drop(columns=["SalePrice","Id"])
    Y = df["SalePrice"]

    colunas_numericas = X.select_dtypes(include=["int64","float64"]).columns
    colunas_categoricas = X.select_dtypes(include="str").columns

    imputer_numerico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="median"))
        ]        
    )

    imputer_categorico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]        
    )

    pre_processador = ColumnTransformer(
        transformers=[
            ("num",imputer_numerico, colunas_numericas),
            ("cat",imputer_categorico,colunas_categoricas)
        ]
    )

    X_treino, X_teste, Y_treino, Y_teste = train_test_split(
        X,
        Y,
        test_size=0.2,
        random_state=42,
    )

    pipeline = Pipeline(
        steps=[
            ("pre_processamento", pre_processador),
            ("modelo",
                TransformedTargetRegressor(
                    regressor=RandomForestRegressor(
                        random_state=42,
                        n_jobs=-1
                    ),
                    func=np.log1p,
                    inverse_func=np.expm1
                )
            )
        ]
    )

    hiperparametros = {
        "modelo__regressor__n_estimators": [25,50,75],
        "modelo__regressor__max_depth": [12,24,48],
        "modelo__regressor__min_samples_split": [2,4,8],
        "modelo__regressor__min_samples_leaf": [2,4,8],
        "modelo__regressor__max_features": ["sqrt", "log2"]
    }

    modelo = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=hiperparametros,
        n_iter=15,
        cv=3,
        scoring="neg_root_mean_squared_log_error",
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    tempo_comeco = time.perf_counter()
    modelo.fit(X_treino,Y_treino)

    validar_ajuste(modelo.best_estimator_,X_treino,Y_treino,X_teste,Y_teste)
    tempo_fim = time.perf_counter()

    tempo_total = tempo_fim - tempo_comeco

    Y_predito = modelo.predict(X_teste)
    
    rmsle = root_mean_squared_log_error(Y_teste, Y_predito)
    rmse = root_mean_squared_error(Y_teste, Y_predito)
    mae = mean_absolute_error(Y_teste, Y_predito)
    r2 = r2_score(Y_teste, Y_predito)

    salvar_modelo(modelo,"./modelos/random-forest.joblib")

    criar_csv_comparacao("random_florest",rmsle,rmse,mae,r2,tempo_total)

    print(f"melhores hiperparametros: {modelo.best_params_}")
    print(f"melhor score: {modelo.best_score_:.4f}")
    print(modelo.best_estimator_.named_steps["pre_processamento"].get_feature_names_out())

    print(f"RMSLE: {rmsle:.4f} - {comparar_baseline("RMSLE",rmsle)}")
    print(f"RMSE: $ {rmse:.4f} - {comparar_baseline("RMSE",rmse)}")
    print(f"MAE: $ {mae:.4f} - {comparar_baseline("MAE",mae)}")
    print(f"R2: {r2:.4f} - {comparar_baseline("R2",r2)}")

print("random florest")
treinar_random_forest(df_treino)

random florest
Fitting 3 folds for each of 15 candidates, totalling 45 fits



===== VALIDAÇÃO DO MODELO =====
R² treino: 0.9214
R² teste:  0.8161
RMSE treino: 21417.3862
RMSE teste:  34474.4855
Possível OVERFITTING
melhores hiperparametros: {'modelo__regressor__n_estimators': 50, 'modelo__regressor__min_samples_split': 2, 'modelo__regressor__min_samples_leaf': 2, 'modelo__regressor__max_features': 'sqrt', 'modelo__regressor__max_depth': 48}
melhor score: -0.1595
['num__MSSubClass' 'num__LotFrontage' 'num__LotArea' 'num__OverallQual'
 'num__OverallCond' 'num__YearBuilt' 'num__YearRemodAdd' 'num__MasVnrArea'
 'num__BsmtFinSF1' 'num__BsmtFinSF2' 'num__BsmtUnfSF' 'num__TotalBsmtSF'
 'num__1stFlrSF' 'num__2ndFlrSF' 'num__LowQualFinSF' 'num__GrLivArea'
 'num__BsmtFullBath' 'num__BsmtHalfBath' 'num__FullBath' 'num__HalfBath'
 'num__BedroomAbvGr' 'num__KitchenAbvGr' 'num__TotRmsAbvGrd'
 'num__Fireplaces' 'num__GarageYrBlt' 'num__GarageCars' 'num__GarageArea'
 'num__WoodDeckSF' 'num__OpenPorchSF' 'num__EnclosedPorch'
 'num__3SsnPorch' 'num__ScreenPorch' 'num__PoolArea' 

# XGBoost

In [35]:

def treinar_xgboost(df:pd.DataFrame):
    X = df.drop(columns=["SalePrice","Id"])
    Y = df["SalePrice"]

    colunas_numericas = X.select_dtypes(include="number").columns
    colunas_categoricas = X.select_dtypes(include="str").columns

    imputer_numerico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="median"))
        ]        
    )

    imputer_categorico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]        
    )

    pre_processador = ColumnTransformer(
        transformers=[
            ("num",imputer_numerico, colunas_numericas),
            ("cat",imputer_categorico,colunas_categoricas)
        ]
        # remainder="passthrough"
    )

    X_treino, X_teste, Y_treino, Y_teste = train_test_split(
        X,
        Y,
        test_size=0.2,
        random_state=42,
    )

    pipeline = Pipeline(
        steps=[
            ("pre-processamento",pre_processador),
            ("modelo",
                TransformedTargetRegressor(
                    regressor=XGBRegressor(
                        random_state=42
                    ),
                    func=np.log1p,
                    inverse_func=np.expm1
                )
            )
        ]
    )
    
    # hiperparametros = {
    #     "modelo__regressor__n_estimators": [200, 500, 800],
    #     "modelo__regressor__learning_rate": [0.01,0.05, 0.1],
    #     "modelo__regressor__max_depth": [2,8,12],
    #     "modelo__regressor__min_child_weight": [2,4,6],
    #     "modelo__regressor__subsample": [0.2, 0.5, 1.0],
    #     "modelo__regressor__colsample_bytree": [0.2,0.5,1.0],
    #     "modelo__regressor__gamma": [0.1,0.5,1.0],
    #     "modelo__regressor__reg_lambda": [0.1,0.5,1.0]
    # }

    hiperparametros = {
        "modelo__regressor__n_estimators": [200,400,800],
        "modelo__regressor__learning_rate": [0.1,0.5,1.0],
        "modelo__regressor__max_depth": [2,4,8],
        "modelo__regressor__min_child_weight": [2,4,8],
        "modelo__regressor__subsample": [0.1,0.5,1.0],
        "modelo__regressor__colsample_bytree": [0,1,0.5,1.0],
        "modelo__regressor__gamma": [0.1,0.5,1.0],
        "modelo__regressor__reg_lambda": [0.1,0.5,1.0]
    }


    modelo = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=hiperparametros,
        n_iter=15,
        cv=5,
        scoring="neg_root_mean_squared_log_error",
        verbose=1,
        random_state=42,
        n_jobs=-1
    )
    
    tempo_comeco = time.perf_counter()
    modelo.fit(X_treino,Y_treino)
    validar_ajuste(modelo.best_estimator_,X_treino,Y_treino,X_teste,Y_teste)
    tempo_fim = time.perf_counter()

    tempo_total = tempo_fim - tempo_comeco

    Y_predito = modelo.predict(X_teste)
    
    rmsle = root_mean_squared_log_error(Y_teste, Y_predito)
    rmse = root_mean_squared_error(Y_teste, Y_predito)
    mae = mean_absolute_error(Y_teste, Y_predito)
    r2 = r2_score(Y_teste, Y_predito)

    salvar_modelo(modelo,"./modelos/xgboost.joblib")
    criar_csv_comparacao("xgboost",rmsle,rmse,mae,r2,tempo_total)

    print(f"melhores hiperparametros: {modelo.best_params_}")
    print(f"melhor score: {modelo.best_score_:.4f}")

    print(f"RMSLE: {rmsle:.4f} - {comparar_baseline("RMSLE",rmsle)}")
    print(f"RMSE: $ {rmse:.4f} - {comparar_baseline("RMSE",rmse)}")
    print(f"MAE: $ {mae:.4f} - {comparar_baseline("MAE",mae)}")
    print(f"R2: {r2:.4f} - {comparar_baseline("R2",r2)}")

print("xgboost")
treinar_xgboost(df_treino)

xgboost
Fitting 5 folds for each of 15 candidates, totalling 75 fits

===== VALIDAÇÃO DO MODELO =====
R² treino: 0.9439
R² teste:  0.9074
RMSE treino: 18101.6992
RMSE teste:  24461.0312
Modelo aparentemente bem ajustado
melhores hiperparametros: {'modelo__regressor__subsample': 0.5, 'modelo__regressor__reg_lambda': 1.0, 'modelo__regressor__n_estimators': 800, 'modelo__regressor__min_child_weight': 4, 'modelo__regressor__max_depth': 2, 'modelo__regressor__learning_rate': 0.1, 'modelo__regressor__gamma': 0.1, 'modelo__regressor__colsample_bytree': 0.5}
melhor score: -0.1320
RMSLE: 0.1291 - True
RMSE: $ 24461.0312 - True
MAE: $ 14850.6641 - True
R2: 0.9074 - True


# CatBoost

In [ ]:
def treinar_catboost(df: pd.DataFrame):

    X = df.drop(columns=["SalePrice", "Id"])
    y = df["SalePrice"]

    colunas_numericas = X.select_dtypes(include="number").columns.tolist()

    colunas_categoricas = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

    imputer_numerico = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]
    )

    imputer_categorico = Pipeline(
        steps=[
            ("imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="missing"
                )
            )
        ]
    )

    pre_processador = ColumnTransformer(
        transformers=[
            ("num", imputer_numerico, colunas_numericas),
            ("cat", imputer_categorico, colunas_categoricas),
        ],
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    X_treino, X_teste, Y_treino, Y_teste = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    pipeline = Pipeline(
        steps=[
            ("pre_processamento", pre_processador),
            ("modelo",
                TransformedTargetRegressor(
                    regressor=CatBoostRegressor(
                        verbose=0,
                        random_state=42,
                        early_stopping_rounds=50,
                        loss_function="RMSE"
                    ),
                    func=np.log1p,
                    inverse_func=np.expm1
                )
            )
        ]
    )

    hiperparametros = {
        "modelo__regressor__iterations": [100,500,1000],
        "modelo__regressor__learning_rate": [0.1,0.5,1.0],
        "modelo__regressor__depth": [2,5],
        "modelo__regressor__l2_leaf_reg": [1,5,8],
        "modelo__regressor__random_strength": [1,5,10],
        "modelo__regressor__min_data_in_leaf": [1,5],
        "modelo__regressor__bagging_temperature": [0,3],
        "modelo__regressor__loss_function": ["RMSE"]
    }

    modelo = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=hiperparametros,
        n_iter=15,
        cv=5,
        scoring="neg_mean_squared_log_error",
        verbose=1,
        random_state=42,
        n_jobs=2
    )

    tempo_comeco = time.perf_counter()

    modelo.fit(
        X_treino,
        Y_treino,
        modelo__cat_features=colunas_categoricas
    )

    tempo_fim = time.perf_counter()
    tempo_total = tempo_fim - tempo_comeco

    Y_predito = modelo.predict(X_teste)

    validar_ajuste(modelo.best_estimator_,X_treino,Y_treino,X_teste,Y_teste)
    
    rmsle = root_mean_squared_log_error(Y_teste, Y_predito)
    rmse = root_mean_squared_error(Y_teste, Y_predito)
    mae = mean_absolute_error(Y_teste, Y_predito)
    r2 = r2_score(Y_teste, Y_predito)

    salvar_modelo(modelo, "./modelos/catboost.joblib")
    
    criar_csv_comparacao("catboost",rmsle,rmse,mae,r2,tempo_total)

    print(f"Melhores hiperparâmetros: {modelo.best_params_}")
    print(f"Melhor score: {modelo.best_score_:.4f}")

    print(f"RMSLE: {rmsle:.4f} - {comparar_baseline("RMSLE",rmsle)}")
    print(f"RMSE: $ {rmse:.4f} - {comparar_baseline("RMSE",rmse)}")
    print(f"MAE: $ {mae:.4f} - {comparar_baseline("MAE",mae)}")
    print(f"R2: {r2:.4f} - {comparar_baseline("R2",r2)}")

print("catboost")
treinar_catboost(df_treino)

catboost
Fitting 5 folds for each of 15 candidates, totalling 75 fits


/home/samir/Área de trabalho/Codigos/ames-housing-aprendizado-maquina/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
50 fits failed out of a total of 75.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "/home/samir/Área de trabalho/Codigos/ames-housing-aprendizado-maquina/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/samir/Área de trabalho/Codigos/ames-housing-aprendizado-maquina/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, 


===== VALIDAÇÃO DO MODELO =====
R² treino: 0.9586
R² teste:  0.8845
RMSE treino: 15541.5856
RMSE teste:  27320.6137
Modelo aparentemente bem ajustado
Melhores hiperparâmetros: {'modelo__regressor__random_strength': 5, 'modelo__regressor__min_data_in_leaf': 5, 'modelo__regressor__loss_function': 'RMSE', 'modelo__regressor__learning_rate': 0.5, 'modelo__regressor__l2_leaf_reg': 1, 'modelo__regressor__iterations': 50, 'modelo__regressor__depth': 5, 'modelo__regressor__bagging_temperature': 0}
Melhor score: -31258.2800
RMSLE: 0.1391 - True
RMSE: $ 27320.6137 - True
MAE: $ 16965.5702 - True
R2: 0.8845 - True


# Resultados


In [37]:
df_resultados = pd.read_csv("./comparacao_modelos_treino.csv")

print(df_resultados.sort_values(by="rmsle",ascending=True).to_string())

                         modelo    rmsle       rmse        mae       r2  tempo_s
2                       xgboost  0.12905   24461.03   14850.66  0.91000  13.8339
3                      catboost  0.13913   27320.61   16965.57  0.88000  21.2382
1                random_florest  0.15723   34474.49   20072.86  0.82000  16.2517
0  linear regression (BASELINE)  0.17543  36,061.40  22,186.99  0.83046      NaN
